# Prithvi WxC Downscaling with CORDEX Data: Model Inference

This notebook walks through running inference with a fine-tuned Prithvi downscaling model on CORDEX NZ data.

To replicate the results show in this notebook please download the required files from CORDEX-ML Bench (https://zenodo.org/records/17517423) repository

You need `git lfs` installed to download large files

---

## Setup

Python >= 3.10 is required

Make sure that your current working directory is `granite-wxc/`

In [ ]:
!pwd

In [ ]:
import os
os.chdir('/mnt/data2/kyo/granite-wxc/examples/CORDEX_ML')


If your current directory is not `granite-wxc/`, change it using the following command:

```bash
%cd <local>/granite-wxc/
```

Replace `<local>` with the appropriate path prefix 

In [ ]:
!pip install -q git+https://github.com/NASA-IMPACT/Prithvi-WxC.git

In [ ]:
!pip install -q h5netcdf matplotlib wget pyyaml xarray scipy torch tqdm pysteps cartopy

In [ ]:
import logging
import warnings
logging.disable(logging.CRITICAL)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from contextlib import nullcontext
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Dataset

from cordex_dataset import CordexDownscaleDataset
from granitewxc.utils.config import get_config
from granitewxc.models.model import get_finetune_model_UNET, get_finetune_model

Configure the backends, PyTorch states, and random seeds to standardize the RNG for random crops in this example

In [ ]:
torch.jit.enable_onednn_fusion(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True
    torch.cuda.manual_seed(42)
torch.manual_seed(42)
np.random.seed(42)

It is possible to use a cpu or gpu/s to generate inferences. Based on avaiablity of a `cuda:gpu`, we set the device that the model uses

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Configuration File

The model is configured using `YAML` files.

In these files, you specify:
- Paths to the input data  
- Locations of the pretrained weights  

To ensure compatibility with the provided weights during inference, keep the model configuration consistent with the original definitions 

In [ ]:
import os
from pathlib import Path

from run_utils import (
    assert_no_eccc_reference,
    load_run_manifest,
    resolve_checkpoint_path,
)

RUNS_ROOT = Path("/mnt/data2/kyo/granite-wxc/examples/CORDEX_ML/runs/nz_finetune").resolve()
RUN_NAME = os.environ.get("NZ_FINETUNE_RUN")
if RUN_NAME:
    run_dir = RUNS_ROOT / RUN_NAME
else:
    candidates = sorted(
        (p for p in RUNS_ROOT.iterdir() if p.is_dir()),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No fine-tune runs found under {RUNS_ROOT}."
        )
    run_dir = candidates[0]
    RUN_NAME = run_dir.name

manifest = load_run_manifest(run_dir)
resolved_config_path = Path(manifest["config_snapshot"]).resolve()
os.chdir('/mnt/data2/kyo/granite-wxc')
config = get_config(str(resolved_config_path))
assert_no_eccc_reference(resolved_config_path)
print(f"Using fine-tune run '{RUN_NAME}' located at {run_dir}")


In [ ]:
from pathlib import Path
PREDICTOR_ROOT = Path(
    "/mnt/data2/kyo/granite-wxc/granite-geospatial-wxc-downscaling/CORDEX/NZ_domain/test/historical_perfect/predictors")


if not PREDICTOR_ROOT.exists():
    raise FileNotFoundError(f"Perfect predictors not found: {PREDICTOR_ROOT}")
config.data.test_predictor_paths = sorted(str(p) for p in PREDICTOR_ROOT.glob("*.nc"))
print(f"Using perfect predictors from {PREDICTOR_ROOT} ({len(config.data.test_predictor_paths)} files)")


In [ ]:
preproc_cache = Path(manifest["preproc_dir"]).resolve()
print(f"Preprocessed predictor cache (shared with fine-tune run): {preproc_cache}")


## Dataloader 

We reuse the same regridded CORDEX sample for testing to showcase the end-to-end pipeline.

In [ ]:
def _level_suffix(level):
    if isinstance(level, str):
        value = level
    else:
        value = str(level)
    return value[:-2] if value.endswith(".0") else value

predictor_variables = [
    f"{var}_{_level_suffix(level)}"
    for var in config.data.input_vars
    for level in config.data.input_levels
]
target_variables = list(config.data.output_vars)

def _build_base_dataset(predictor_paths, target_paths):
    return CordexDownscaleDataset(
        predictor_files=predictor_paths,
        target_files=target_paths,
        orography_file=config.data.static_path,
        predictor_variables=predictor_variables,
        target_variables=target_variables,
    )

class CordexWrappedDataset(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        sample = self.base[idx]
        static = sample["x"][-1:].clone()
        x = sample["x"][:-1].clone()
        return {"x": x, "y": sample["y"], "static_x": static, "static_y": static}

def build_dataloader(predictor_paths, target_paths):
    dataset = CordexWrappedDataset(_build_base_dataset(predictor_paths, target_paths))
    return DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=config.dl_num_workers,
        pin_memory=(device.type == "cuda"),
    )

test_dl = build_dataloader(
    config.data.test_predictor_paths, config.data.test_target_paths
)

len(test_dl)

## Model Initialization

We provide **2** different model architectures `UNET-like` and `CONV` 

Both architectures include:  
1. **Patch Embedding**: Extracts shallow features from the input data  
2. **Feature Extraction**: Utilizes the Prithvi backbone to extract deeper features  

The key difference is that the UNET-like version incorporates **static high-resolution data** into the model

In this notebook, we use the **UNET-like** version

To switch to the **CONV** model, update the configuration file accordingly and use `get_finetune_model(config)`

In [ ]:
model = get_finetune_model_UNET(config)

We can now load the pretrained weights

In [ ]:
from pathlib import Path

ckpt_path = resolve_checkpoint_path(manifest, preference="best")
assert_no_eccc_reference(ckpt_path)
weights = torch.load(str(ckpt_path), map_location='cpu')['model']

model_state = model.state_dict()
weights_have_module_prefix = all(key.startswith('module.') for key in weights.keys())
model_expects_module_prefix = all(key.startswith('module.') for key in model_state.keys())

if model_expects_module_prefix and not weights_have_module_prefix:
    weights = weights.__class__((f"module.{key}", value) for key, value in weights.items())
elif weights_have_module_prefix and not model_expects_module_prefix:
    prefix_len = len('module.')
    weights = weights.__class__((key[prefix_len:], value) for key, value in weights.items())

model.load_state_dict(weights, strict=True)
model.to(device)

skip_offload_devices = []
if device.type == "cuda" and torch.cuda.device_count() > 1:
    skip_offload_devices = [torch.device(f"cuda:{idx}") for idx in range(1, torch.cuda.device_count())]
    if skip_offload_devices and hasattr(model, "set_skip_activation_devices"):
        model.set_skip_activation_devices(skip_offload_devices)
        print(f"--> Offloading skip activations to GPUs: {skip_offload_devices}")

print(f"Loaded fine-tuned weights from {ckpt_path}")


### Inference

The model is now ready for inference. We run inference for one batch (batch_size=1).

In [ ]:
with torch.no_grad():
    model.eval()
    
    batch = next(iter(test_dl))
    batch = {k: v.to(device) for k, v in batch.items()}
    autocast_enabled = device.type == "cuda"
    autocast_dtype = torch.bfloat16 if (autocast_enabled and torch.cuda.is_bf16_supported()) else torch.float16
    autocast_ctx = torch.cuda.amp.autocast(dtype=autocast_dtype, enabled=autocast_enabled) if autocast_enabled else nullcontext()
    
    with autocast_ctx:
        out = model(batch)

    inputs = batch['x']
    targets = batch['y']
    outputs = out

In [ ]:
inputs.shape, targets.shape, outputs.shape

inputs.shape, targets.shape, outputs.shape

In [ ]:
### Plotting

#We visualise one of the output channels to inspect the prediction quality.

In [ ]:
var_names = list(config.data.output_vars)
var_idx = 0  # pr
sample_idx = 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(targets[sample_idx, var_idx].cpu(), cmap='coolwarm')
axes[0].set_title(f'Target {var_names[var_idx]}')
axes[0].axis("off")
axes[1].imshow(outputs[sample_idx, var_idx].cpu(), cmap='coolwarm')
axes[1].set_title(f'Prediction {var_names[var_idx]}')
axes[1].axis("off")
plt.show()

In [ ]:

# Generate full-field predictions for the entire test set and store them in NetCDF format
prediction_output_path = (run_dir / "predictions" / f"{RUN_NAME}_predictions.nc")
prediction_output_path.parent.mkdir(parents=True, exist_ok=True)

base_dataset = test_dl.dataset.base
time_dim = base_dataset.time_dim or "time"
lat_name = base_dataset.fine_lat_name
lon_name = base_dataset.fine_lon_name
target_vars = list(base_dataset.target_vars)

def _run_full_inference(dataloader, model, device):
    predictions = []
    autocast_enabled = device.type == "cuda"
    autocast_dtype = (
        torch.bfloat16 if (autocast_enabled and torch.cuda.is_bf16_supported()) else torch.float16
    )

    def autocast_context():
        return (
            torch.cuda.amp.autocast(dtype=autocast_dtype, enabled=autocast_enabled)
            if autocast_enabled
            else nullcontext()
        )

    with torch.no_grad():
        model.eval()
        for batch in tqdm(dataloader, desc="Running inference", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast_context():
                out = model(batch)
            predictions.append(out.detach().cpu())
    return torch.cat(predictions, dim=0)

full_outputs = _run_full_inference(test_dl, model, device)
outputs_np = full_outputs.numpy()



In [ ]:
with xr.open_mfdataset(
    base_dataset.target_paths,
    combine="nested",
    concat_dim=time_dim,
    data_vars="minimal",
    coords="minimal",
    compat="override",
    engine="h5netcdf",
) as template_ds:
    template_attrs = dict(template_ds.attrs)
    target_attrs = {
        name: dict(template_ds[name].attrs) for name in target_vars if name in template_ds.data_vars
    }
    metadata_ds = template_ds.drop_vars(target_vars).load()

prediction_ds = metadata_ds.copy()
coords = {
    time_dim: metadata_ds[time_dim],
    lat_name: metadata_ds[lat_name],
    lon_name: metadata_ds[lon_name],
}

for idx, name in enumerate(target_vars):
    prediction_ds[name] = xr.DataArray(
        outputs_np[:, idx],
        dims=(time_dim, lat_name, lon_name),
        coords=coords,
        attrs=target_attrs.get(name, {}),
    )

prediction_ds.attrs.update(template_attrs)
prediction_ds.to_netcdf(prediction_output_path, engine="h5netcdf")
print(f"Saved predictions to {prediction_output_path}")


In [ ]:
import pickle

pickle_path = run_dir / "predictions" / f"{RUN_NAME}_predictions.pkl"
with open(pickle_path, "wb") as handle:
    pickle.dump(outputs_np, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved raw predictions array to {pickle_path}")
